# 🗣️ SpeakSteps ML Pipeline
## Predicting Cue Needs for Broca's Aphasia Therapy

This notebook trains machine learning models to predict whether a user will need a cue for their next therapy question, enabling adaptive difficulty adjustment.

**Models:** Decision Tree, Logistic Regression  
**Target:** `need_cue_next` (binary classification)


## 1. Setup & Imports


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Model persistence
import joblib

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("✅ All libraries imported successfully!")


## 2. Load Data


In [ ]:
# For Google Colab: Upload the CSV file first
# from google.colab import files
# uploaded = files.upload()  # Uncomment to upload interactively

# Load the dataset
df = pd.read_csv('synth_speaksteps.csv')

print(f"📊 Dataset Shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")
df.head()


In [ ]:
# Quick data overview
print("\n📈 Data Types:")
print(df.dtypes)
print("\n📉 Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n📊 Basic Statistics:")
df.describe()


## 3. Data Preprocessing & Feature Engineering


In [ ]:
# Create a copy for processing
data = df.copy()

# ============================================
# 3.1 Handle Missing Values
# ============================================
# Fill missing cue-related columns (when no cue was given)
data['cue_type'] = data['cue_type'].fillna('none')
data['cue_wait_seconds'] = data['cue_wait_seconds'].fillna(0)

print("✅ Missing values handled")


In [ ]:
# ============================================
# 3.2 Create Target Variable: need_cue_next
# ============================================
# Rule: If response_time > threshold OR incorrect answer → user needs cue next
RESPONSE_TIME_THRESHOLD = 30  # seconds

data['need_cue_next'] = (
    (data['response_time_seconds'] > RESPONSE_TIME_THRESHOLD) |
    (data['correct'] == 0)
).astype(int)

print(f"\n🎯 Target Variable Distribution:")
print(data['need_cue_next'].value_counts())
print(f"\n📊 Class Balance: {data['need_cue_next'].mean()*100:.1f}% positive (need cue)")


In [ ]:
# ============================================
# 3.3 Create Features
# ============================================

# --- Difficulty Flag (binary) ---
data['difficulty_flag'] = (data['difficulty_label'] == 'hard').astype(int)

# --- Device Mobile Flag ---
data['device_mobile_flag'] = (data['device_type'] == 'mobile').astype(int)

# --- Time of Day Bucket ---
# Extract hour from presented_at_iso
data['presented_hour'] = pd.to_datetime(data['presented_at_iso']).dt.hour

def get_time_bucket(hour):
    """Categorize hour into time-of-day buckets"""
    if 5 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 17:
        return 'afternoon'
    elif 17 <= hour < 21:
        return 'evening'
    else:
        return 'night'

data['time_of_day'] = data['presented_hour'].apply(get_time_bucket)

# One-hot encode time_of_day
time_dummies = pd.get_dummies(data['time_of_day'], prefix='time')
data = pd.concat([data, time_dummies], axis=1)

print("✅ Time of day features created")
print(data['time_of_day'].value_counts())


In [ ]:
# --- Module Encoding (one-hot) ---
module_dummies = pd.get_dummies(data['module'], prefix='module')
data = pd.concat([data, module_dummies], axis=1)

# --- Category Encoding (one-hot) ---
category_dummies = pd.get_dummies(data['category'], prefix='cat')
data = pd.concat([data, category_dummies], axis=1)

# --- Question Type Encoding (ordinal) ---
le_question = LabelEncoder()
data['question_type_encoded'] = le_question.fit_transform(data['question_type'])

# --- Cue Type Encoding (ordinal) ---
le_cue = LabelEncoder()
data['cue_type_encoded'] = le_cue.fit_transform(data['cue_type'])

print("✅ Categorical encodings complete")
print(f"\nQuestion types: {list(le_question.classes_)}")
print(f"Cue types: {list(le_cue.classes_)}")


In [ ]:
# ============================================
# 3.4 Select Final Features
# ============================================

FEATURE_COLUMNS = [
    # Core numeric features
    'response_time_seconds',
    'cue_given',
    'cue_stage',
    'hint_count',
    'difficulty_flag',
    'device_mobile_flag',
    'therapist_assigned_level',

    # Encoded categorical
    'question_type_encoded',
    'cue_type_encoded',

    # One-hot: time of day
    'time_morning',
    'time_afternoon',
    'time_evening',
    'time_night',

    # One-hot: module
    'module_comprehension',
    'module_writing',

    # One-hot: category
    'cat_animals',
    'cat_body_parts',
    'cat_clothing',
    'cat_food',
]

TARGET_COLUMN = 'need_cue_next'

# Verify all columns exist
missing_cols = [col for col in FEATURE_COLUMNS if col not in data.columns]
if missing_cols:
    print(f"⚠️ Missing columns: {missing_cols}")
else:
    print(f"✅ All {len(FEATURE_COLUMNS)} feature columns found")

# Create feature matrix and target vector
X = data[FEATURE_COLUMNS].copy()
y = data[TARGET_COLUMN].copy()

print(f"\n📐 Feature Matrix Shape: {X.shape}")
print(f"🎯 Target Vector Shape: {y.shape}")


In [ ]:
# Preview features
X.head(10)


## 4. Train/Test Split


In [ ]:
# 80/20 split, stratified by target
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"📊 Training Set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"📊 Test Set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\n🎯 Train Target Distribution:")
print(y_train.value_counts())
print(f"\n🎯 Test Target Distribution:")
print(y_test.value_counts())


In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled for Logistic Regression")


## 5. Model Training

### 5.1 Decision Tree Classifier


In [ ]:
# Hyperparameter tuning for Decision Tree
MAX_DEPTH_OPTIONS = [3, 5, 7, 10, 15, None]

dt_results = []

for depth in MAX_DEPTH_OPTIONS:
    dt = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    )
    # Cross-validation
    cv_scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='f1')
    dt_results.append({
        'max_depth': depth,
        'mean_f1': cv_scores.mean(),
        'std_f1': cv_scores.std()
    })
    print(f"max_depth={str(depth):>4} | F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Find best depth
best_dt_config = max(dt_results, key=lambda x: x['mean_f1'])
print(f"\n🏆 Best max_depth: {best_dt_config['max_depth']} (F1={best_dt_config['mean_f1']:.4f})")


In [ ]:
# Train final Decision Tree with best hyperparameters
dt_model = DecisionTreeClassifier(
    max_depth=best_dt_config['max_depth'],
    random_state=RANDOM_STATE,
    class_weight='balanced',
    min_samples_split=10,
    min_samples_leaf=5
)
dt_model.fit(X_train, y_train)

# Predictions
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

print("✅ Decision Tree trained!")


### 5.2 Logistic Regression


In [ ]:
# Train Logistic Regression with balanced class weights
lr_model = LogisticRegression(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    max_iter=1000,
    solver='lbfgs'
)
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print("✅ Logistic Regression trained!")


## 6. Model Evaluation


In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    """Calculate and display evaluation metrics"""
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }
    return metrics

# Evaluate both models
dt_metrics = evaluate_model(y_test, y_pred_dt, y_prob_dt, 'Decision Tree')
lr_metrics = evaluate_model(y_test, y_pred_lr, y_prob_lr, 'Logistic Regression')

# Display comparison table
results_df = pd.DataFrame([dt_metrics, lr_metrics])
results_df = results_df.set_index('Model')

print("\n" + "="*60)
print("📊 MODEL COMPARISON")
print("="*60)
print(results_df.round(4).to_string())


In [ ]:
# Classification Reports
print("\n" + "="*60)
print("📋 DECISION TREE - Classification Report")
print("="*60)
print(classification_report(y_test, y_pred_dt, target_names=['No Cue Needed', 'Cue Needed']))

print("\n" + "="*60)
print("📋 LOGISTIC REGRESSION - Classification Report")
print("="*60)
print(classification_report(y_test, y_pred_lr, target_names=['No Cue Needed', 'Cue Needed']))


In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision Tree Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Cue', 'Cue Needed'],
            yticklabels=['No Cue', 'Cue Needed'])
axes[0].set_title('Decision Tree\nConfusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Logistic Regression Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['No Cue', 'Cue Needed'],
            yticklabels=['No Cue', 'Cue Needed'])
axes[1].set_title('Logistic Regression\nConfusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Confusion matrices saved to 'confusion_matrices.png'")


In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

# Decision Tree ROC
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
ax.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {dt_metrics["ROC-AUC"]:.3f})',
        linewidth=2, color='#2196F3')

# Logistic Regression ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_metrics["ROC-AUC"]:.3f})',
        linewidth=2, color='#4CAF50')

# Diagonal (random classifier)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', alpha=0.5)

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ ROC curves saved to 'roc_curves.png'")


## 7. Feature Importance Analysis


In [ ]:
# Decision Tree Feature Importance
dt_importance = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*60)
print("🌳 DECISION TREE - Feature Importance")
print("="*60)
print(dt_importance.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(dt_importance)))
bars = ax.barh(dt_importance['Feature'], dt_importance['Importance'], color=colors[::-1])
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Decision Tree Feature Importance', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, dt_importance['Importance']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance_dt.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Logistic Regression Coefficients
lr_coef = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Coefficient': lr_model.coef_[0],
    'Abs_Coefficient': np.abs(lr_model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("\n" + "="*60)
print("📈 LOGISTIC REGRESSION - Feature Coefficients")
print("="*60)
print(lr_coef[['Feature', 'Coefficient']].to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#4CAF50' if c > 0 else '#F44336' for c in lr_coef['Coefficient']]
bars = ax.barh(lr_coef['Feature'], lr_coef['Coefficient'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient', fontsize=12)
ax.set_title('Logistic Regression Coefficients\n(Green=Increases Cue Need, Red=Decreases)',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('coefficients_lr.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Visualize Decision Tree (limited depth for readability)
fig, ax = plt.subplots(figsize=(20, 12))
plot_tree(
    dt_model,
    feature_names=FEATURE_COLUMNS,
    class_names=['No Cue', 'Cue Needed'],
    filled=True,
    rounded=True,
    ax=ax,
    max_depth=3,  # Limit for visualization
    fontsize=9
)
ax.set_title('Decision Tree Visualization (max_depth=3 for clarity)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree_viz.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Save Best Model


In [ ]:
# Determine best model based on F1 score
if dt_metrics['F1 Score'] >= lr_metrics['F1 Score']:
    best_model = dt_model
    best_model_name = 'Decision Tree'
    best_metrics = dt_metrics
    # Save scaler as None for DT (doesn't need scaling)
    model_bundle = {
        'model': dt_model,
        'scaler': None,
        'feature_columns': FEATURE_COLUMNS,
        'model_type': 'DecisionTree'
    }
else:
    best_model = lr_model
    best_model_name = 'Logistic Regression'
    best_metrics = lr_metrics
    model_bundle = {
        'model': lr_model,
        'scaler': scaler,
        'feature_columns': FEATURE_COLUMNS,
        'model_type': 'LogisticRegression'
    }

# Save the model
joblib.dump(model_bundle, 'model.joblib')

print(f"\n" + "="*60)
print(f"🏆 BEST MODEL: {best_model_name}")
print("="*60)
print(f"F1 Score: {best_metrics['F1 Score']:.4f}")
print(f"ROC-AUC: {best_metrics['ROC-AUC']:.4f}")
print(f"\n✅ Model saved to 'model.joblib'")


In [ ]:
# Verify saved model loads correctly
loaded_bundle = joblib.load('model.joblib')
print(f"✅ Model loaded successfully!")
print(f"   Type: {loaded_bundle['model_type']}")
print(f"   Features: {len(loaded_bundle['feature_columns'])} columns")
print(f"   Scaler: {'Included' if loaded_bundle['scaler'] else 'Not needed'}")


## 9. Recommendations for Improvement

### 🔧 Hyperparameter Tuning Recommendations

#### Decision Tree
- **GridSearchCV/RandomizedSearchCV:** Explore broader parameter space:
  - `max_depth`: [3, 5, 7, 10, 15, 20, None]
  - `min_samples_split`: [2, 5, 10, 20]
  - `min_samples_leaf`: [1, 2, 5, 10]
  - `criterion`: ['gini', 'entropy']
  - `max_features`: ['sqrt', 'log2', None]

#### Logistic Regression
- **Regularization strength (C):** Try [0.001, 0.01, 0.1, 1, 10, 100]
- **Penalty type:** Compare 'l1' (sparse) vs 'l2' (ridge)
- **Solver:** 'saga' supports both l1/l2 and is scalable

#### Advanced Models to Consider
- **Random Forest:** Ensemble of trees, reduces overfitting
- **Gradient Boosting (XGBoost/LightGBM):** Often best performance
- **Neural Networks:** For complex non-linear patterns

---

### 📊 Data Collection Improvements

1. **Temporal Features:**
   - Session duration and fatigue indicators
   - Days since last session (practice gap)
   - Performance trend over last N questions

2. **User History Features:**
   - Rolling accuracy (last 5/10/20 questions)
   - Category-specific success rates
   - Historical cue dependency per user

3. **Engagement Metrics:**
   - Pause patterns before answering
   - Number of answer changes/corrections
   - App interaction patterns (scrolling, zooming)

4. **Clinical Features:**
   - Aphasia severity score (if available)
   - Time since stroke/injury
   - Co-occurring conditions

5. **Better Label Engineering:**
   - Use actual next-question cue data (not simulated)
   - Consider multi-class: no cue, light cue, heavy cue
   - Incorporate therapist feedback as ground truth

---

### 🎯 Model Deployment Considerations

1. **A/B Testing:** Compare model recommendations vs. current system
2. **Confidence Thresholds:** Only act on high-confidence predictions
3. **User Override:** Allow therapists to override model suggestions
4. **Continuous Learning:** Retrain periodically with new data
5. **Explainability:** Use SHAP values for individual predictions


In [ ]:
# Example: Using GridSearchCV for better tuning
print("📝 Example: GridSearchCV for Decision Tree")
print("-" * 50)

param_grid = {
    'max_depth': [5, 7, 10, 15],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\n🏆 Best Parameters: {grid_search.best_params_}")
print(f"🏆 Best CV F1 Score: {grid_search.best_score_:.4f}")

# Evaluate on test set
y_pred_best = grid_search.predict(X_test)
print(f"\n📊 Test Set Performance:")
print(f"   F1 Score: {f1_score(y_test, y_pred_best):.4f}")
print(f"   Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")


## 10. Summary


In [ ]:
print("\n" + "="*70)
print("🗣️ SPEAKSTEPS ML PIPELINE - SUMMARY")
print("="*70)
print(f"""
📊 Dataset: {len(df)} samples loaded
🔧 Features: {len(FEATURE_COLUMNS)} engineered features
📈 Train/Test Split: 80/20 (stratified)

🌳 Decision Tree Results:
   - Accuracy: {dt_metrics['Accuracy']:.4f}
   - F1 Score: {dt_metrics['F1 Score']:.4f}
   - ROC-AUC:  {dt_metrics['ROC-AUC']:.4f}

📈 Logistic Regression Results:
   - Accuracy: {lr_metrics['Accuracy']:.4f}
   - F1 Score: {lr_metrics['F1 Score']:.4f}
   - ROC-AUC:  {lr_metrics['ROC-AUC']:.4f}

🏆 Best Model: {best_model_name} (saved as model.joblib)

📁 Output Files:
   - model.joblib (trained model bundle)
   - confusion_matrices.png
   - roc_curves.png
   - feature_importance_dt.png
   - coefficients_lr.png
   - decision_tree_viz.png
""")
print("="*70)
print("✅ Pipeline complete!")


---

## 11. TensorFlow / TFLite Model for Mobile Deployment

This section builds a simple neural network mimicking Logistic Regression, then converts it to TensorFlow Lite for on-device inference.

**Expected Results:**
- TFLite model size: **10–80 KB** (single dense layer)
- Inference latency on mid-range Android: **1–2 ms**


In [ ]:
#==============================================================================
# SPEAKSTEPS - TENSORFLOW / TFLITE MODEL
# Google Colab Ready Script
#==============================================================================

import os
import numpy as np

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Sklearn metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")


In [ ]:
#==============================================================================
# 1. VERIFY PREPROCESSED DATA EXISTS
#==============================================================================

try:
    # Check if variables exist from previous cells
    _ = X_train.shape
    _ = y_train.shape
    _ = X_test.shape
    _ = y_test.shape
    
    print("✅ Preprocessed data found!")
    print(f"   X_train shape: {X_train.shape}")
    print(f"   y_train shape: {y_train.shape}")
    print(f"   X_test shape:  {X_test.shape}")
    print(f"   y_test shape:  {y_test.shape}")
    
    # Convert to numpy if pandas DataFrame
    if hasattr(X_train, 'values'):
        X_train_np = X_train.values.astype('float32')
        X_test_np = X_test.values.astype('float32')
    else:
        X_train_np = np.array(X_train).astype('float32')
        X_test_np = np.array(X_test).astype('float32')
    
    if hasattr(y_train, 'values'):
        y_train_np = y_train.values.astype('float32')
        y_test_np = y_test.values.astype('float32')
    else:
        y_train_np = np.array(y_train).astype('float32')
        y_test_np = np.array(y_test).astype('float32')
    
    INPUT_DIM = X_train_np.shape[1]
    print(f"   Input dimension: {INPUT_DIM}")
    
except NameError as e:
    print("❌ ERROR: Preprocessed data not found!")
    print("   Please run the preprocessing cells first (Sections 1-4).")
    print(f"   Missing: {e}")
    raise SystemExit("Data not available. Run preprocessing cells first.")


In [ ]:
#==============================================================================
# 2. BUILD KERAS MODEL (Logistic Regression Architecture)
#==============================================================================

# Clear any existing models
tf.keras.backend.clear_session()

# Build model mimicking Logistic Regression
# Single Dense layer with sigmoid = Logistic Regression
model = Sequential([
    Input(shape=(INPUT_DIM,), name='input_features'),
    Dense(
        units=1,
        activation='sigmoid',
        kernel_regularizer=l2(0.01),
        name='logistic_output'
    )
], name='SpeakSteps_LogisticRegression')

# Model summary
model.summary()


In [ ]:
#==============================================================================
# 3. COMPILE MODEL
#==============================================================================

model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("✅ Model compiled!")
print("   Loss: binary_crossentropy")
print("   Optimizer: Adam (lr=0.001)")
print("   Metrics: accuracy")


In [ ]:
#==============================================================================
# 4. TRAIN MODEL
#==============================================================================

EPOCHS = 20  # Adjustable
VALIDATION_SPLIT = 0.2
BATCH_SIZE = 32

print(f"\n🏋️ Training for {EPOCHS} epochs...")
print(f"   Validation split: {VALIDATION_SPLIT*100:.0f}%")
print(f"   Batch size: {BATCH_SIZE}")
print("-" * 60)

history = model.fit(
    X_train_np,
    y_train_np,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("\n✅ Training complete!")


In [ ]:
#==============================================================================
# 5. EVALUATE MODEL
#==============================================================================

# Get predictions
y_train_pred_prob = model.predict(X_train_np, verbose=0)
y_test_pred_prob = model.predict(X_test_np, verbose=0)

# Convert probabilities to binary predictions
y_train_pred = (y_train_pred_prob >= 0.5).astype(int).flatten()
y_test_pred = (y_test_pred_prob >= 0.5).astype(int).flatten()

# Calculate metrics
train_accuracy = accuracy_score(y_train_np, y_train_pred)
test_accuracy = accuracy_score(y_test_np, y_test_pred)
test_precision = precision_score(y_test_np, y_test_pred)
test_recall = recall_score(y_test_np, y_test_pred)
test_f1 = f1_score(y_test_np, y_test_pred)

print("\n" + "=" * 60)
print("📊 MODEL EVALUATION")
print("=" * 60)
print(f"\n🎯 Train Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"🎯 Test Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"\n📈 Test Metrics:")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1 Score:  {test_f1:.4f}")


In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('keras_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training history saved to 'keras_training_history.png'")


In [ ]:
#==============================================================================
# 6. SAVE MODEL
#==============================================================================

# 6a. Save as TensorFlow SavedModel format
SAVED_MODEL_DIR = 'saved_model'

model.save(SAVED_MODEL_DIR)
print(f"✅ SavedModel saved to: {SAVED_MODEL_DIR}/")

# List saved files
for root, dirs, files in os.walk(SAVED_MODEL_DIR):
    level = root.replace(SAVED_MODEL_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")


In [ ]:
# 6b. Convert to TensorFlow Lite (Standard FP32)
print("\n" + "=" * 60)
print("📱 CONVERTING TO TFLITE")
print("=" * 60)

# Standard FP32 conversion
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
tflite_model = converter.convert()

# Save FP32 model
TFLITE_MODEL_PATH = 'model.tflite'
with open(TFLITE_MODEL_PATH, 'wb') as f:
    f.write(tflite_model)

tflite_size_kb = os.path.getsize(TFLITE_MODEL_PATH) / 1024
print(f"\n✅ Standard TFLite model saved: {TFLITE_MODEL_PATH}")
print(f"   Size: {tflite_size_kb:.2f} KB")


In [ ]:
# 6c. Convert with Float16 Quantization (optional, smaller model)
converter_quant = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]
converter_quant.target_spec.supported_types = [tf.float16]

tflite_model_quant = converter_quant.convert()

# Save quantized model
TFLITE_QUANT_PATH = 'model_float16.tflite'
with open(TFLITE_QUANT_PATH, 'wb') as f:
    f.write(tflite_model_quant)

tflite_quant_size_kb = os.path.getsize(TFLITE_QUANT_PATH) / 1024
print(f"\n✅ Float16 Quantized TFLite model saved: {TFLITE_QUANT_PATH}")
print(f"   Size: {tflite_quant_size_kb:.2f} KB")
print(f"   Size reduction: {(1 - tflite_quant_size_kb/tflite_size_kb)*100:.1f}%")


In [ ]:
#==============================================================================
# 7. VERIFY TFLITE MODEL
#==============================================================================

print("\n" + "=" * 60)
print("🔍 TFLITE MODEL VERIFICATION")
print("=" * 60)

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

# Get input/output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\n📋 Model Details:")
print(f"   Input shape:  {input_details[0]['shape']}")
print(f"   Input dtype:  {input_details[0]['dtype']}")
print(f"   Output shape: {output_details[0]['shape']}")
print(f"   Output dtype: {output_details[0]['dtype']}")

# Run sample inference with first row of X_test
sample_input = X_test_np[0:1].astype(np.float32)

# Set input tensor
interpreter.set_tensor(input_details[0]['index'], sample_input)

# Run inference
interpreter.invoke()

# Get output
tflite_prediction = interpreter.get_tensor(output_details[0]['index'])
tflite_class = 1 if tflite_prediction[0][0] >= 0.5 else 0

print(f"\n🧪 Sample Inference (first test row):")
print(f"   Input features: {sample_input.shape}")
print(f"   Raw prediction: {tflite_prediction[0][0]:.6f}")
print(f"   Predicted class: {tflite_class} ({'Cue Needed' if tflite_class == 1 else 'No Cue'})")
print(f"   Actual label:    {int(y_test_np[0])} ({'Cue Needed' if y_test_np[0] == 1 else 'No Cue'})")


In [ ]:
# Benchmark TFLite inference time
import time

NUM_RUNS = 100
inference_times = []

for _ in range(NUM_RUNS):
    start = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], sample_input)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]['index'])
    inference_times.append((time.perf_counter() - start) * 1000)  # Convert to ms

avg_inference_ms = np.mean(inference_times)
std_inference_ms = np.std(inference_times)

print(f"\n⏱️ Inference Benchmark ({NUM_RUNS} runs):")
print(f"   Average: {avg_inference_ms:.3f} ms")
print(f"   Std Dev: {std_inference_ms:.3f} ms")
print(f"   Min:     {min(inference_times):.3f} ms")
print(f"   Max:     {max(inference_times):.3f} ms")


In [ ]:
#==============================================================================
# 8. FINAL SUMMARY
#==============================================================================

print("\n" + "=" * 70)
print("📱 TFLITE DEPLOYMENT SUMMARY")
print("=" * 70)
print(f"""
🗂️ Output Files:
   - saved_model/          (TensorFlow SavedModel directory)
   - model.tflite          (Standard FP32: {tflite_size_kb:.2f} KB)
   - model_float16.tflite  (Float16 Quantized: {tflite_quant_size_kb:.2f} KB)

📊 Model Performance:
   - Test Accuracy:  {test_accuracy*100:.2f}%
   - Test F1 Score:  {test_f1:.4f}
   - Test Precision: {test_precision:.4f}
   - Test Recall:    {test_recall:.4f}

⚡ Expected On-Device Performance:
   - TFLite Size: 10–80 KB (single dense layer)
   - Inference Latency: 1–2 ms on mid-range Android devices
   - Measured (Colab): {avg_inference_ms:.3f} ms average

📝 Integration Notes:
   - Use TensorFlow Lite Android/iOS SDK
   - Input: {INPUT_DIM} float32 features
   - Output: Single float32 probability (0-1)
   - Threshold: >= 0.5 → "Cue Needed"
""")
print("=" * 70)
print("✅ TFLite conversion complete!")
